In [1]:
import pandas as pd
from pathlib import Path

In [2]:
# ── Paths ──────────────────────────────────────────────────────────────────
BASE_DIR   = Path(r'c:\Users\Usuario\Documents\Monroe County ORRI')
TABLES_DIR = BASE_DIR / 'agreementsExtractedTables' / 'Excel files man'
OUTPUT_XLS = BASE_DIR / 'agreementsExtractedTables' / 'All_Agreements_Combined.xlsx'

print(f'Source : {TABLES_DIR}')
print(f'Output : {OUTPUT_XLS}')

Source : c:\Users\Usuario\Documents\Monroe County ORRI\agreementsExtractedTables\Excel files man
Output : c:\Users\Usuario\Documents\Monroe County ORRI\agreementsExtractedTables\All_Agreements_Combined.xlsx


In [3]:
# ── Standardised column names (canonical order) ────────────────────────────
STANDARD_COLUMNS = [
    'Sublessor',
    'Sublessee',
    'Date of Sublease',
    'Sublease Ref (Book/Page)',
    'Original Lessor',
    'Original Lessee',
    'Date of Lease',
    'Lease Ref (Book/Page)',   # standardised from various 'Cross Ref' variants
    'Lease No.',
    'Source File',             # ← added: name of the source Excel file
]

# Map any non-standard header names found in source files to their canonical name
HEADER_ALIASES = {
    'Lease Ref (Cross Ref)' : 'Lease Ref (Book/Page)',
    'Lease Ref(Cross Ref)'  : 'Lease Ref (Book/Page)',
    'Lease Reference'       : 'Lease Ref (Book/Page)',
    'Cross Reference'       : 'Lease Ref (Book/Page)',
    'Lease Number'          : 'Lease No.',
    'Lease #'               : 'Lease No.',
}

print('Standard columns defined:', STANDARD_COLUMNS)

Standard columns defined: ['Sublessor', 'Sublessee', 'Date of Sublease', 'Sublease Ref (Book/Page)', 'Original Lessor', 'Original Lessee', 'Date of Lease', 'Lease Ref (Book/Page)', 'Lease No.', 'Source File']


In [4]:
# ── Read, normalise and stack every Excel file ─────────────────────────────
frames = []

for xlsx in sorted(TABLES_DIR.glob('*.xlsx')):
    if xlsx.name.startswith('~$') or xlsx.name == OUTPUT_XLS.name:
        continue

    try:
        df = pd.read_excel(xlsx, header=0, dtype=str)
    except Exception as e:
        print(f'  [ERR] {xlsx.name}: {e}')
        continue

    # 1. Strip whitespace from column names
    df.columns = [str(c).strip() for c in df.columns]

    # 2. Apply header aliases
    df.rename(columns=HEADER_ALIASES, inplace=True)

    # 3. Add source-file column
    df['Source File'] = xlsx.stem

    # 4. Keep only standard columns (add missing ones as empty)
    for col in STANDARD_COLUMNS:
        if col not in df.columns:
            df[col] = ''
    df = df[STANDARD_COLUMNS]

    # 5. Drop completely empty data rows
    data_cols = [c for c in STANDARD_COLUMNS if c != 'Source File']
    df = df.dropna(how='all', subset=data_cols)
    df = df[~(df[data_cols].apply(lambda r: r.str.strip().eq('')).all(axis=1))]

    frames.append(df)
    print(f'  [OK]  {xlsx.name:55s}  {len(df):>4d} rows')

combined = pd.concat(frames, ignore_index=True)
print(f'\nTotal rows combined: {len(combined)}')

  [OK]  AEU to AEUM_294_646.xlsx                                  195 rows
  [OK]  Byers to AEU_287-784.xlsx                                  18 rows
  [OK]  Byers to AEUM_293_506.xlsx                                  2 rows
  [OK]  Clift to AEUM_292_984.xlsx                                  1 rows
  [OK]  D&D to AEU_289_939.xlsx                                     8 rows
  [OK]  Gardner to AEU_287_794.xlsx                                 1 rows
  [OK]  Howell to AEUM_295_392.xlsx                                33 rows
  [OK]  Howell to AEUM_302_663.xlsx                                 2 rows
  [OK]  Kroll to AEUM_298_858.xlsx                                  1 rows
  [OK]  Kroll to AEUM_301_11.xlsx                                   1 rows
  [OK]  Mahoney to AEUM_299_695.xlsx                                1 rows
  [OK]  Speedy Turtle to Dorado_290_540.xlsx                        1 rows
  [OK]  Speedy Turtle_Dorado to AEU_290_546.xlsx                    1 rows
  [OK]  Templeton to AEUM

In [5]:
# ── Light cleaning ─────────────────────────────────────────────────────────

# Strip leading/trailing whitespace from all string cells
combined = combined.apply(
    lambda col: col.str.strip() if col.dtype == object else col
)

# Replace bare 'nan' strings (artefact of dtype=str read) with empty string
combined.replace('nan', '', inplace=True)

# Normalise date columns: convert to YYYY-MM-DD where possible
for date_col in ['Date of Sublease', 'Date of Lease']:
    converted = pd.to_datetime(combined[date_col], errors='coerce')
    mask = converted.notna()
    combined.loc[mask, date_col] = converted[mask].dt.strftime('%Y-%m-%d')

print('Sample of combined table:')
combined.head(10)

Sample of combined table:


,Sublessor,Sublessee,Date of Sublease,Sublease Ref (Book/Page),Original Lessor,Original Lessee,Date of Lease,Lease Ref (Book/Page),Lease No.,Source File
0,Darrell R. Cline,"HG Energy, LLC",2011-07-18,OR 216-974,"BRIGGS, ROY D. ETAL",DARRELL R. CLINE,1980-08-02,LR 117-295,OH00176-00,AEU to AEUM_294_646
1,Darrell R Cline,"HG Energy, LLC",2011-07-18,OR 216-974,"FRANCES J.L AND CLINE, B.F",BENTZ OIL,1924-01-02,LR 71-178,OH00176-00,AEU to AEUM_294_646
2,Darrell R Cline,"HG Energy, LLC",2011-07-18,OR 216-974,"PIATT, FORREST B.","THE CROW OIL GAS, AND COAL",1920-12-17,LR 93-219,OH00177-00,AEU to AEUM_294_646
3,Darrell R. Cline,"HG Energy, LLC",2011-07-08,OR 216-974,DOUGHERTY D.C. ETUX,BONNIE S CLINE,1981-11-12,LR 121-145,OH00178-00,AEU to AEUM_294_646
4,"Whitacre Enterprise, Inc.","HG Energy, LLC",2011-02-22,OR 217-705,ETAL,N.T. STAUDT,1935-03-30,LR 82-18,OH00236-00,AEU to AEUM_294_646
5,Same as above,"HG Energy, LLC",2011-12-22,OR 217-705,EARLE PEARL AND GRIFFIN,EVANS JOHN ROY,1987-09-15,LR 106-216,OH00502-00,AEU to AEUM_294_646
6,"Whitacre Enterprises, Inc.","HG Energy, LLC",2011-06-02,OR 224-636,"JOHNSON, CLARICE AND ED",THE CARTER OIL COMPANY,1910-06-30,LR 71-503,OH00100-01,AEU to AEUM_294_646
7,K.L.J. Oil Company,"HG Energy, LLC",2011-06-02,OR 224-636,JOHNSON EDWARD A,THE PARSONS & SWEENEY CO,2012-02-19,LR 57-584,OH00100-02,AEU to AEUM_294_646
8,Clearfork Oil Company,"HG Energy, LLC",2011-06-02,OR 224-636,CLARICE AND S.,K.L.J. INC.,1985-04-02,LR 30-012,OH00101-00,AEU to AEUM_294_646
9,Buckeye Oil Company,"HG Energy, LLC",2011-05-02,OR 224-636,"WILLISON, ALVA AND MARIE ETAL",WHITACRE OIL CO,1980-08-05,LR 116-870,OH00102-00,AEU to AEUM_294_646


In [6]:
# ── Save to Excel ──────────────────────────────────────────────────────────
with pd.ExcelWriter(OUTPUT_XLS, engine='openpyxl') as writer:
    combined.to_excel(writer, index=False, sheet_name='All Agreements')

    # Auto-fit column widths
    ws = writer.sheets['All Agreements']
    for col_cells in ws.columns:
        max_len = max(
            (len(str(cell.value)) if cell.value is not None else 0)
            for cell in col_cells
        )
        ws.column_dimensions[col_cells[0].column_letter].width = min(max_len + 2, 60)

print(f'Saved -> {OUTPUT_XLS}')
print(f'Rows : {len(combined)}')
print(f'Cols : {list(combined.columns)}')

Saved -> c:\Users\Usuario\Documents\Monroe County ORRI\agreementsExtractedTables\All_Agreements_Combined.xlsx
Rows : 445
Cols : ['Sublessor', 'Sublessee', 'Date of Sublease', 'Sublease Ref (Book/Page)', 'Original Lessor', 'Original Lessee', 'Date of Lease', 'Lease Ref (Book/Page)', 'Lease No.', 'Source File']
